# Tutorial 5 — Evaluating extraction quality

An extraction pipeline you cannot measure is a pipeline you cannot improve.
This project ships two things to measure it with: an **LLM judge** that scores
any extraction against the source text, and a **human-annotated corpus** of 36
papers that lets you ask the harder question — *does the judge agree with a
domain expert?*

## What you'll learn

1. How to run `DspyGeneralSynthesisJudge` on an extraction and read its seven
   score dimensions
2. That the judge actually discriminates — we degrade an extraction on purpose
   and watch the scores fall
3. How the `annotations/` corpus is laid out, and how to load the full
   extractor × judge grid into a DataFrame
4. How well four LLM judges agree with human scores — computed live, not quoted
5. What the numbers do and don't justify

## Prerequisites

- The package installed (`uv sync && uv pip install -e .`)
- **Runtime:** ~5 min. **Cost:** three LLM calls in Part A (fractions of a
  cent); Parts B–D are free and offline.

## Step 0 — API keys and your `.env` file

Nothing in this project takes an API key as a function argument. Keys live in a
single `.env` file at the repository root, get loaded into the process
environment once per session, and LiteLLM/DSPy read them from there. That means
**your keys never appear in notebook code, notebook outputs, or git history**.

### Create your `.env`

From the repository root:

```bash
cp .env.example .env
```

Then open `.env` and fill in the keys you need — one per line, no quotes and no
spaces around `=`:

```
GEMINI_API_KEY=AIza...
ANTHROPIC_API_KEY=sk-ant-...
```

`.env` is git-ignored, so it never gets committed.

### Keys used by this tutorial

| Key | What it unlocks | Needed here? |
|-----|-----------------|--------------|
| `GEMINI_API_KEY` | The extractor and judge used in Part A (direct providers) | yes, for Part A |
| `OPENROUTER_API_KEY` | The same two models, routed through OpenRouter | only on that path |

Where to get them: **Gemini** (free tier is enough for this tutorial) at
[aistudio.google.com](https://aistudio.google.com/app/apikey), **Anthropic** at
[console.anthropic.com](https://console.anthropic.com/), **Mistral** at
[console.mistral.ai](https://console.mistral.ai/), **OpenRouter** at
[openrouter.ai/keys](https://openrouter.ai/keys).

> Parts B, C and D read files that ship with the repository and make **no API
> calls at all** — you can skip Part A entirely if you have no key.

The next cell loads `.env` and reports which keys arrived — it prints only the
key *length*, never the value, so the output is safe to share.

In [ ]:
import os
import logging

from dotenv import find_dotenv, load_dotenv

# dspy warns whenever .forward(...) is called directly instead of module(...);
# every extractor/judge in this pipeline is invoked via .forward() by design
# (see transformers/base.py's ExtractorInterface), so this is expected noise.
logging.getLogger("dspy.primitives.module").setLevel(logging.ERROR)

# find_dotenv walks up from the working directory, so this works whether you
# started Jupyter at the repo root or inside this folder.
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

# Filled in after the provider flag is set in the next cell; Parts B-D need
# no keys at all.
REQUIRED_KEYS = {"GEMINI_API_KEY": "extraction + judging in Part A"}

OPTIONAL_KEYS = {"OPENROUTER_API_KEY": "set USE_OPENROUTER = True to use it"}


def report_keys(required, optional):
    """Print which API keys .env provided, without revealing their values."""
    print(f".env loaded from: {env_path or 'NOT FOUND'}\n")
    missing = []
    for name, purpose in {**required, **optional}.items():
        value = os.getenv(name)
        is_required = name in required
        if value:
            status = f"set ({len(value)} chars)"
        elif is_required:
            status = "MISSING"
            missing.append(name)
        else:
            status = "not set"
        tag = "required" if is_required else "optional"
        print(f"  {name:<28} {status:<16} [{tag}] {purpose}")
    if missing:
        raise RuntimeError(
            "Missing required key(s): "
            + ", ".join(missing)
            + ". Copy .env.example to .env at the repository root and fill "
            "them in, then re-run this cell."
        )
    print("\nAll required keys are present.")


report_keys(REQUIRED_KEYS, OPTIONAL_KEYS)

### Direct providers or OpenRouter

Two ways to reach the models. **Direct** uses one key per provider
(`GEMINI_API_KEY`); **OpenRouter** uses a single `OPENROUTER_API_KEY` and any
model id from [openrouter.ai/models](https://openrouter.ai/models). Set the flag
in the next cell — nothing else in the notebook changes.

In [ ]:
# --- Provider -----------------------------------------------------------
USE_OPENROUTER = False  # True routes every LLM call through OpenRouter

OPENROUTER_API_BASE = "https://openrouter.ai/api/v1"
DIRECT_MODEL = "gemini-3.0-flash"  # an alias from LLM_REGISTRY
OPENROUTER_MODEL = "google/gemini-3-flash-preview"  # any openrouter.ai model


def build_lm(system_prompt: str = "", **model_kwargs):
    """Return a cost-tracking LM, honouring USE_OPENROUTER."""
    from llm_synthesis.utils.dspy_utils import get_llm_from_name
    from llm_synthesis.utils.llms import SystemPrefixedLM

    if USE_OPENROUTER:
        return SystemPrefixedLM(
            system_prompt,
            f"openrouter/{OPENROUTER_MODEL}",
            api_base=OPENROUTER_API_BASE,
            api_key=os.environ["OPENROUTER_API_KEY"],
            **model_kwargs,
        )
    return get_llm_from_name(
        DIRECT_MODEL, model_kwargs=model_kwargs, system_prompt=system_prompt
    )


print("model:", OPENROUTER_MODEL if USE_OPENROUTER else DIRECT_MODEL)

## Part A — Run the judge

The judge takes three inputs — the source text, the extraction as JSON, and the
material name — and returns a `GeneralSynthesisEvaluation` with a 1–5 score plus
written reasoning for each of seven dimensions.

We need something to judge, so we extract from a **synthetic paper** written for
this tutorial (not a real publication).

In [ ]:
DEMO_PAPER = """# Hydrothermal synthesis of Co-doped ZnO nanorods

*Synthetic example written for the LeMat-Synth tutorials - not a real paper.*

## Experimental

Zn(NO3)2 - 6H2O (2.97 g, 99.9%, Sigma-Aldrich) and Co(NO3)2 - 6H2O (0.29 g)
were dissolved in 80 mL of deionised water under magnetic stirring for 30 min at
room temperature. Hexamethylenetetramine (1.40 g) was then added and the
solution stirred for a further 15 min.

The mixture was transferred into a 100 mL Teflon-lined stainless steel autoclave
and held at 95 C for 6 h, then cooled naturally to room temperature.

The precipitate was collected by centrifugation at 6000 rpm, washed three times
with deionised water and once with ethanol, and dried in a vacuum oven at 60 C
for 12 h. The powder was finally calcined in air at 450 C for 2 h at a heating
rate of 5 C/min to give Zn0.95Co0.05O nanorods.
"""

MATERIAL = "Zn0.95Co0.05O"
print(f"{len(DEMO_PAPER)} characters, target material: {MATERIAL}")

In [ ]:
from llm_synthesis.transformers.synthesis_extraction import (
    DspySynthesisExtractor,
    make_dspy_synthesis_extractor_signature,
)

extractor = DspySynthesisExtractor(
    signature=make_dspy_synthesis_extractor_signature(),
    lm=build_lm(temperature=0.0, max_tokens=16000),
)

synthesis = extractor.forward(input=(DEMO_PAPER, MATERIAL))

print(f"Method: {synthesis.synthesis_method}")
print(f"Type:   {synthesis.target_compound_type}")
print(f"Steps:  {len(synthesis.steps)}")
print(f"Precursors: {[m.name for m in synthesis.starting_materials]}")

In [ ]:
import json

from llm_synthesis.metrics.judge.general_synthesis_judge import (
    DspyGeneralSynthesisJudge,
    make_general_synthesis_judge_signature,
)

# A low but non-zero temperature is the convention for judges in this repo:
# deterministic enough to be reproducible, loose enough to reason.
judge = DspyGeneralSynthesisJudge(
    signature=make_general_synthesis_judge_signature(),
    lm=build_lm(temperature=0.1, max_tokens=4096),
)


def show_scores(evaluation, label):
    """Print every *_score field of a GeneralSynthesisEvaluation."""
    scores = evaluation.scores
    print(f"=== {label} ===")
    for field in type(scores).model_fields:
        if field.endswith("_score"):
            name = field.replace("_score", "").replace("_", " ")
            print(f"  {name:<26} {getattr(scores, field)}/5.0")
    print(f"  confidence: {evaluation.confidence_level}")


evaluation = judge.forward(
    (DEMO_PAPER, json.dumps(synthesis.model_dump()), MATERIAL)
)
show_scores(evaluation, "as extracted")
print("\nOverall reasoning:")
print(evaluation.scores.overall_reasoning)

### Does the judge actually discriminate?

A judge that says "4.5/5" to everything is worthless. The cheapest sanity check
is to damage the extraction on purpose — here we throw away every process step
and precursor — and confirm the scores fall where they should (structural
completeness and process steps), not uniformly.

In [ ]:
degraded = synthesis.model_copy(deep=True)
degraded.steps = []
degraded.starting_materials = []

degraded_evaluation = judge.forward(
    (DEMO_PAPER, json.dumps(degraded.model_dump()), MATERIAL)
)
show_scores(degraded_evaluation, "steps and precursors removed")

print("\nDrop per dimension:")
for field in type(evaluation.scores).model_fields:
    if not field.endswith("_score"):
        continue
    before = getattr(evaluation.scores, field)
    after = getattr(degraded_evaluation.scores, field)
    if before is None or after is None:
        continue
    name = field.replace("_score", "").replace("_", " ")
    print(
        f"  {name:<26} {before:>4.2f} -> {after:>4.2f}  ({after - before:+.2f})"
    )

## Part B — The human annotation corpus

`annotations/` holds 36 papers where a domain expert read the paper and scored
what the models produced. The layout per paper:

```
annotations/<paper_id>/
├── result.json         extractor × judge grid: 4 extractors, each scored by 4 LLM judges
└── result_human.json   the expert's own recipe plus their score for each extractor
```

In `result_human.json`, `materials[i]["evaluations"]` is **positionally aligned**
with the top-level `extractor_order` list — entry 0 is the expert's score of
extractor 0's output, and so on.

The directory is **read/append-only**: new annotations arrive by pull request
from the Streamlit annotator app, never by editing in place.

In [ ]:
import json
from pathlib import Path

import pandas as pd

ANNOTATIONS_DIR = Path("../../../annotations")

SCORE_COLUMNS = [
    "structural_completeness_score",
    "material_extraction_score",
    "process_steps_score",
    "equipment_extraction_score",
    "conditions_extraction_score",
    "semantic_accuracy_score",
    "format_compliance_score",
    "overall_score",
]


def _scores(evaluation):
    """Dig the scores dict out of one evaluation record, tolerating nulls."""
    return ((evaluation or {}).get("evaluation") or {}).get("scores") or {}


def load_annotation_scores(annotations_dir):
    """Flatten every human and LLM judge score into one long DataFrame."""
    rows, unreadable = [], []

    for paper_dir in sorted(p for p in annotations_dir.iterdir() if p.is_dir()):
        try:
            human = json.loads((paper_dir / "result_human.json").read_text())
            machine = json.loads((paper_dir / "result.json").read_text())
        except (OSError, json.JSONDecodeError) as exc:
            # A corrupt file must not take the whole corpus down with it.
            unreadable.append((paper_dir.name, type(exc).__name__))
            continue

        paper_id = human.get("paper_id", paper_dir.name)

        # Human scores: aligned by position with extractor_order.
        for material in human["materials"]:
            for extractor, evaluation in zip(
                human["extractor_order"], material["evaluations"]
            ):
                scores = _scores(evaluation)
                rows.append(
                    {
                        "paper": paper_id,
                        "material": material["material_name"],
                        "extractor": extractor,
                        "judge": "human",
                        **{c: scores.get(c) for c in SCORE_COLUMNS},
                    }
                )

        # LLM scores: every (extractor, judge) combination.
        for entry in machine:
            for material in entry["materials"]:
                for evaluation in material.get("evaluations", []):
                    scores = _scores(evaluation)
                    rows.append(
                        {
                            "paper": paper_id,
                            "material": material["material"],
                            "extractor": entry["synth_llm"],
                            "judge": evaluation.get("judge_llm"),
                            **{c: scores.get(c) for c in SCORE_COLUMNS},
                        }
                    )

    return pd.DataFrame(rows), unreadable


scores_df, unreadable = load_annotation_scores(ANNOTATIONS_DIR)

print(f"{len(scores_df):,} score rows from {scores_df.paper.nunique()} papers")
print(f"extractors: {sorted(scores_df.extractor.unique())}")
print(f"judges:     {sorted(scores_df.judge.unique())}")
for name, error in unreadable:
    print(f"  SKIPPED {name}: {error}")

In [ ]:
# How much of the grid is actually filled in? Human annotation is expensive and
# the corpus is partially scored - that shapes everything that follows.
coverage = (
    scores_df.assign(scored=scores_df.overall_score.notna())
    .groupby("judge")
    .scored.agg(["sum", "size"])
    .rename(columns={"sum": "scored", "size": "rows"})
)
coverage["coverage"] = (coverage.scored / coverage.rows * 100).round(1)
coverage

## Part C — Do the LLM judges agree with the humans?

We pair every human-scored extraction with the LLM judges' scores for the *same*
`(paper, material, extractor)` and compare.

Two honest caveats, visible in the numbers below:

- **Exact name matching loses pairs.** Extractors write material names
  differently (`1%Ru/CeO2` vs `1 wt% Ru/CeO₂`), so an exact join drops some
  rows. The evaluation scripts under `examples/scripts/evaluation/` use fuzzy
  matching (`eval_utils.find_best_matches`, `normalize_material_name`) to
  recover them.
- **The sample is small.** Only part of the human-scored corpus survives the
  exact-name join, and those pairs fall in a handful of papers — enough to say
  "this judge runs harsh" or "that one passes everything", not "judge A is 3%
  better than judge B".

In [ ]:
human_scores = scores_df[scores_df.judge == "human"].dropna(
    subset=["overall_score"]
)
llm_scores = scores_df[scores_df.judge != "human"].dropna(
    subset=["overall_score"]
)

pairs = human_scores.merge(
    llm_scores,
    on=["paper", "material", "extractor"],
    suffixes=("_h", "_l"),
)

print(
    f"{len(pairs)} human/judge pairs recovered from "
    f"{len(human_scores)} human-scored extractions "
    f"(exact material-name match)"
)

In [ ]:
pairs = pairs.assign(
    abs_diff=(pairs.overall_score_h - pairs.overall_score_l).abs()
)

agreement = (
    pairs.groupby("judge_l")
    .agg(
        n=("abs_diff", "size"),
        human_mean=("overall_score_h", "mean"),
        judge_mean=("overall_score_l", "mean"),
        mae=("abs_diff", "mean"),
    )
    .round(2)
)
agreement["bias"] = (agreement.judge_mean - agreement.human_mean).round(2)
agreement["pearson_r"] = pd.Series(
    {
        judge: group.overall_score_h.corr(group.overall_score_l)
        for judge, group in pairs.groupby("judge_l")
    }
).round(2)

print(
    "Agreement with human overall_score (higher r better, |bias| lower better)"
)
agreement

In [ ]:
# Which dimensions do humans and judges disagree about most?
per_criterion = []
for column in SCORE_COLUMNS:
    human_col, judge_col = pairs[f"{column}_h"], pairs[f"{column}_l"]
    both = human_col.notna() & judge_col.notna()
    per_criterion.append(
        {
            "criterion": column.replace("_score", ""),
            "n": int(both.sum()),
            "human_mean": human_col[both].mean(),
            "judge_mean": judge_col[both].mean(),
            "mae": (human_col[both] - judge_col[both]).abs().mean(),
        }
    )

pd.DataFrame(per_criterion).round(2).sort_values("mae", ascending=False)

In [ ]:
import matplotlib.pyplot as plt

judges = sorted(pairs.judge_l.unique())
fig, axes = plt.subplots(
    1, len(judges), figsize=(4 * len(judges), 4), sharey=True
)

for ax, judge_name in zip(axes, judges):
    group = pairs[pairs.judge_l == judge_name]
    ax.scatter(
        group.overall_score_h,
        group.overall_score_l,
        alpha=0.5,
        s=28,
        color="#4C72B0",
    )
    ax.plot([1, 5], [1, 5], "--", color="grey", linewidth=1)
    ax.set_title(judge_name, fontsize=10)
    ax.set_xlabel("human overall")
    ax.set_xlim(1, 5.3)
    ax.set_ylim(1, 5.3)

axes[0].set_ylabel("LLM judge overall")
fig.suptitle("LLM judge vs human (dashed = perfect agreement)")
plt.tight_layout()
plt.show()

### Reading these numbers

Points above the dashed line mean the judge was more generous than the human.
On the current corpus most judges sit *below* the line — they score the same
extractions more harshly than the expert did — while `gemini-3-flash` is pinned
near the ceiling and effectively passes everything. Either way the scores are
compressed into the 4–5 band, humans included, which is why **mean absolute
error is the honest headline metric and correlation looks weak**: with little
variance to explain, `r` is fragile.

Practically:

- Use judge scores as a **filter** (drop the low tail) rather than as a precise
  ranking. That is exactly what the `high_score` config of the published dataset
  does.
- Compare extractors with the **same** judge. Cross-judge comparisons inherit
  each judge's bias.
- For publication-grade numbers use the scripts in
  `examples/scripts/evaluation/`, which add fuzzy material matching and
  intraclass correlation (`calculate_icc_absolute_agreement`) on top of what we
  computed here.


## Part D — Contributing annotations

More human-scored papers is the highest-value contribution to this project.
The workflow:

```bash
streamlit run examples/scripts/data_curation/annotator_app.py
```

The app writes `annotations/<paper_id>/result_human.json`; you then open a pull
request. Because `annotations/` is append-only, contributions never disturb
existing ground truth.

> **Parse defensively anyway.** Every file in the corpus is valid JSON today,
> so `load_annotation_scores` skipped nothing above. Keep its `try/except`
> regardless: a half-written annotation from an in-flight branch should be
> reported and stepped over, not allowed to take the whole analysis down.

## What's next

- **[Tutorial 6 — Customising the ontology](06_customizing_the_ontology.ipynb)**:
  if the judge keeps docking points for something your domain does not care
  about, change the schema rather than the scores.
- **[Tutorial 1 — Explore the LeMat-Synth dataset](01_explore_the_lemat_synth_dataset.ipynb)**:
  the `evaluation` column there is this judge, run at scale.
- **[Tutorial 3 — Batch extraction with the CLI](03_batch_extraction_with_the_cli.ipynb)**:
  `judge_model=` picks which model does the scoring in a production run.
